## 1. Dependencias y herramientas base

In [ ]:
!sudo apt install -y git repo curl python3 build-essential bc bison flex libssl-dev libncurses-dev
!mkdir -p ~/bin && curl https://storage.googleapis.com/git-repo-downloads/repo > ~/bin/repo
!chmod a+x ~/bin/repo
!export PATH="$HOME/bin:$PATH"

!git config --global user.email "you@example.com"
!git config --global user.name "Your Name"


## 2. Descargar el árbol del kernel (repo sync)

In [ ]:
!mkdir -p ~/lamu-kernel
%cd ~/lamu-kernel

!yes | repo init -u https://github.com/moto-mediatek-devs/android_kernel_motorola_lamu_manifest -b lineage-23.2 --depth=1
!repo sync -c -j24 --force-sync --no-clone-bundle --optimized-fetch

## 3. Parche kABI obligatorio (kernel < 6.12)

Sin este parche, habilitar `CONFIG_SYSVIPC`/`CONFIG_IPC_NS` desplaza offsets del `task_struct` y bootloopea por romper kABI. Confirmado inocuo por sí solo (probado con defconfig stock, arrancó bien).

In [ ]:
%cd ~/lamu-kernel
!wget -O kabi_sysvipc.patch https://raw.githubusercontent.com/ravindu644/Droidspaces-OSS/main/Documentation/resources/kernel-patches/GKI/below-kernel-6.12/001.GKI-below-6.12-fix_sysvipc_kabi_6_7_8.patch

%cd ~/lamu-kernel/kernel-6.6
!patch -p1 < ~/lamu-kernel/kabi_sysvipc.patch
!rm -f include/linux/sched.h.orig


## 5. Editar `gki_defconfig` — las 5 opciones, `PID_NS` forzado explícito

`CONFIG_PID_NS=y` se agrega como línea explícita esta vez: en una build anterior resolvió solo en `=y` (Kleaf lo marcó redundante vía `savedefconfig`), pero en otra build volvió a aparecer como `# is not set`. No es confiable dejarlo implícito — se fuerza.

`CONFIG_IPC_NS` sí se dejó fuera de forma consistente en ambos builds y **sí** resolvió en `=y` las dos veces — por ahora se mantiene implícito, pero si en el próximo build también falla, forzarlo igual que `PID_NS`.

**Se commitea el resultado al final del paso**, para que este build tenga un hash de git único y distinguible de builds anteriores con otro contenido de defconfig.

In [5]:
%%bash
cd ~/lamu-kernel/kernel-6.6

python3 - << 'PYEOF'
path = "arch/arm64/configs/gki_defconfig"
with open(path) as f:
    lines = f.readlines()

def ensure_after(lines, anchor, new_line):
    if any(l.strip() == new_line for l in lines):
        return lines
    out = []
    inserted = False
    for l in lines:
        out.append(l)
        if not inserted and l.strip() == anchor:
            out.append(new_line + "\n")
            inserted = True
    if not inserted:
        out.append(new_line + "\n")
    return out

def remove_line(lines, target):
    return [l for l in lines if l.strip() != target]

# quitar la linea '# CONFIG_PID_NS is not set' si existe, antes de forzar =y
lines = remove_line(lines, '# CONFIG_PID_NS is not set')

lines = ensure_after(lines, 'CONFIG_LOCALVERSION="-4k"', 'CONFIG_SYSVIPC=y')
lines = ensure_after(lines, 'CONFIG_SYSVIPC=y', 'CONFIG_POSIX_MQUEUE=y')
lines = ensure_after(lines, 'CONFIG_PCI_ENDPOINT=y', 'CONFIG_DEVTMPFS=y')
lines = ensure_after(lines, 'CONFIG_NAMESPACES=y', 'CONFIG_PID_NS=y')

with open(path, 'w') as f:
    f.writelines(lines)
PYEOF

echo "--- Resultado ---"
grep -n "CONFIG_SYSVIPC\|CONFIG_PID_NS\|CONFIG_IPC_NS\|CONFIG_POSIX_MQUEUE\|CONFIG_DEVTMPFS" arch/arm64/configs/gki_defconfig

# Si Bazel se queja con 'ERROR: savedefconfig does not match...' al compilar,
# el log imprime el diff exacto requerido -- aplicarlo con patch respetando
# la posicion canonica que pide. Si vuelve a marcar PID_NS como redundante y
# lo quita, dejar que savedefconfig gane esta vez y confirmar con extract-ikconfig
# en la seccion 7 si igual resuelve en =y; si no, puede requerir una dependencia
# especifica (revisar 'select PID_NS' en init/Kconfig) en vez de la linea directa.

git add arch/arm64/configs/gki_defconfig
git commit -m "droidspaces: sysvipc+posix_mqueue+devtmpfs+pid_ns explicito" || echo "(nada que commitear, revisar diff arriba)"
git rev-parse --short HEAD


--- Resultado ---
3:CONFIG_SYSVIPC=y
4:CONFIG_POSIX_MQUEUE=y
41:CONFIG_PID_NS=y
319:CONFIG_DEVTMPFS=y
[detached HEAD 25c0187ec] droidspaces: sysvipc+posix_mqueue+devtmpfs+pid_ns explicito
 1 file changed, 4 insertions(+), 1 deletion(-)
25c0187ec


In [6]:
%%bash
cd ~/lamu-kernel/kernel-6.6

# Solo las opciones del bloque "GKI-Exclusive Configuration" de la doc de Droidspaces
declare -A opts=(
  [CONFIG_SYSVIPC]=y
  [CONFIG_POSIX_MQUEUE]=y
  [CONFIG_IPC_NS]=y
  [CONFIG_PID_NS]=y
  [CONFIG_DEVTMPFS]=y
  [CONFIG_NETFILTER_XT_MATCH_ADDRTYPE]=y
)

for opt in "${!opts[@]}"; do
  val="${opts[$opt]}"
  if grep -q "^${opt}=" arch/arm64/configs/gki_defconfig; then
    echo "$opt ya está en =y, sin cambios"
  elif grep -q "^# ${opt} is not set" arch/arm64/configs/gki_defconfig; then
    sed -i "s/^# ${opt} is not set/${opt}=${val}/" arch/arm64/configs/gki_defconfig
    echo "$opt: activado (existía como 'is not set')"
  else
    echo "${opt}=${val}" >> arch/arm64/configs/gki_defconfig
    echo "$opt: agregado al final (no existía)"
  fi
done

echo "--- Resultado ---"
grep -n "CONFIG_SYSVIPC\|CONFIG_PID_NS\|CONFIG_IPC_NS\|CONFIG_POSIX_MQUEUE\|CONFIG_DEVTMPFS\|CONFIG_NETFILTER_XT_MATCH_ADDRTYPE" \
    arch/arm64/configs/gki_defconfig

git add arch/arm64/configs/gki_defconfig
git commit -m "Droidspaces GKI-exclusive config (sin opciones de Docker Swarm)"
git rev-parse --short HEAD

CONFIG_IPC_NS: agregado al final (no existía)
CONFIG_NETFILTER_XT_MATCH_ADDRTYPE: agregado al final (no existía)
CONFIG_SYSVIPC ya está en =y, sin cambios
CONFIG_PID_NS ya está en =y, sin cambios
CONFIG_DEVTMPFS ya está en =y, sin cambios
CONFIG_POSIX_MQUEUE ya está en =y, sin cambios
--- Resultado ---
3:CONFIG_SYSVIPC=y
4:CONFIG_POSIX_MQUEUE=y
41:CONFIG_PID_NS=y
319:CONFIG_DEVTMPFS=y
795:CONFIG_IPC_NS=y
796:CONFIG_NETFILTER_XT_MATCH_ADDRTYPE=y
[detached HEAD 8631171f7] Droidspaces GKI-exclusive config (sin opciones de Docker Swarm)
 1 file changed, 2 insertions(+)
8631171f7


In [7]:
%%bash
cd ~/lamu-kernel/kernel-6.6

# Quitar las 4 opciones que no están en el bloque GKI-Exclusive de la doc
sed -i '/^CONFIG_BRIDGE_NETFILTER=y$/d;/^CONFIG_VXLAN=y$/d;/^CONFIG_IP_VS=y$/d;/^CONFIG_IP_VS_RR=y$/d' \
    arch/arm64/configs/gki_defconfig

echo "--- Resultado ---"
grep -n "CONFIG_SYSVIPC\|CONFIG_PID_NS\|CONFIG_IPC_NS\|CONFIG_POSIX_MQUEUE\|CONFIG_DEVTMPFS\|CONFIG_VXLAN\|CONFIG_BRIDGE_NETFILTER\|CONFIG_IP_VS\|CONFIG_NETFILTER_XT_MATCH_ADDRTYPE" \
    arch/arm64/configs/gki_defconfig

git add arch/arm64/configs/gki_defconfig
git commit -m "revertir opciones de Docker Swarm no presentes en doc de Droidspaces (VXLAN, IP_VS, IP_VS_RR, BRIDGE_NETFILTER)"
git rev-parse --short HEAD

--- Resultado ---
3:CONFIG_SYSVIPC=y
4:CONFIG_POSIX_MQUEUE=y
41:CONFIG_PID_NS=y
319:CONFIG_DEVTMPFS=y
795:CONFIG_IPC_NS=y
796:CONFIG_NETFILTER_XT_MATCH_ADDRTYPE=y
Not currently on any branch.
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   include/linux/sched.h

no changes added to commit (use "git add" and/or "git commit -a")
8631171f7


In [8]:
%cd ~/lamu-kernel/kernel-6.6

/root/lamu-kernel/kernel-6.6


In [9]:
!sed -i '/^CONFIG_PID_NS=y$/d' arch/arm64/configs/gki_defconfig
!grep -n "CONFIG_SYSVIPC\|CONFIG_PID_NS\|CONFIG_IPC_NS\|CONFIG_POSIX_MQUEUE\|CONFIG_DEVTMPFS" arch/arm64/configs/gki_defconfig

3:CONFIG_SYSVIPC=y
4:CONFIG_POSIX_MQUEUE=y
318:CONFIG_DEVTMPFS=y
794:CONFIG_IPC_NS=y


In [10]:
%%bash
cd ~/lamu-kernel/kernel-6.6
git add arch/arm64/configs/gki_defconfig
git commit -m "revertir PID_NS explicito, dejar que Kleaf lo resuelva via dependencia"
git rev-parse --short HEAD

[detached HEAD 9518cc259] revertir PID_NS explicito, dejar que Kleaf lo resuelva via dependencia
 1 file changed, 1 deletion(-)
9518cc259


In [11]:
%%bash
cd ~/lamu-kernel/kernel-6.6

CFG=arch/arm64/configs/gki_defconfig

# 1. Quitar el bloque que se agregó al final (append ciego)
sed -i '/^CONFIG_IPC_NS=y$/d;/^CONFIG_NETFILTER_XT_MATCH_ADDRTYPE=y$/d;/^CONFIG_SYSVIPC=y$/d;/^CONFIG_DEVTMPFS=y$/d;/^CONFIG_POSIX_MQUEUE=y$/d' "$CFG"

# 2. Reinsertar cada opción en su posición canónica (según savedefconfig)
sed -i '/^CONFIG_LOCALVERSION="-4k"$/a CONFIG_SYSVIPC=y\nCONFIG_POSIX_MQUEUE=y' "$CFG"
sed -i '/^CONFIG_NETFILTER_XT_TARGET_TCPMSS=y$/a CONFIG_NETFILTER_XT_MATCH_ADDRTYPE=y' "$CFG"
sed -i '/^CONFIG_PCI_ENDPOINT=y$/a CONFIG_DEVTMPFS=y' "$CFG"

echo "--- Resultado ---"
grep -n "CONFIG_SYSVIPC\|CONFIG_PID_NS\|CONFIG_IPC_NS\|CONFIG_POSIX_MQUEUE\|CONFIG_DEVTMPFS\|CONFIG_NETFILTER_XT_MATCH_ADDRTYPE" "$CFG"

git add "$CFG"
git commit -m "Droidspaces GKI config en posicion canonica (savedefconfig), sin CONFIG_IPC_NS explicito"
git rev-parse --short HEAD

--- Resultado ---
3:CONFIG_SYSVIPC=y
4:CONFIG_POSIX_MQUEUE=y
203:CONFIG_NETFILTER_XT_MATCH_ADDRTYPE=y
319:CONFIG_DEVTMPFS=y
[detached HEAD 98e7be363] Droidspaces GKI config en posicion canonica (savedefconfig), sin CONFIG_IPC_NS explicito
 1 file changed, 1 insertion(+), 2 deletions(-)
98e7be363


## 6. Compilar

In [ ]:
%cd ~/lamu-kernel
!./build_mgk_64_k66.sh


## 7. Verificación decisiva: config REAL embebido en el kernel compilado

Antes de tocar boot.img. Confirmar que las 5 opciones (incluida `PID_NS`, ahora forzada) están en `=y`.

In [14]:
%cd ~/lamu-kernel
!kernel-6.6/scripts/extract-ikconfig out/mgk_64_k66/dist/Image.gz | \
  grep -E "CONFIG_(SYSVIPC|PID_NS|IPC_NS|POSIX_MQUEUE|DEVTMPFS)"


/root/lamu-kernel
CONFIG_SYSVIPC=y
CONFIG_SYSVIPC_SYSCTL=y
CONFIG_SYSVIPC_COMPAT=y
CONFIG_POSIX_MQUEUE=y
CONFIG_POSIX_MQUEUE_SYSCTL=y
CONFIG_IPC_NS=y
CONFIG_PID_NS=y
CONFIG_DEVTMPFS=y
# CONFIG_DEVTMPFS_MOUNT is not set
# CONFIG_DEVTMPFS_SAFE is not set


In [21]:
!kernel-6.6/scripts/extract-ikconfig out/mgk_64_k66/dist/Image.gz | \
  grep -E "CONFIG_(SYSVIPC|PID_NS|IPC_NS|POSIX_MQUEUE|DEVTMPFS|BRIDGE_NETFILTER|VXLAN|NETFILTER_XT_MATCH_ADDRTYPE)"

CONFIG_SYSVIPC=y
CONFIG_SYSVIPC_SYSCTL=y
CONFIG_SYSVIPC_COMPAT=y
CONFIG_POSIX_MQUEUE=y
CONFIG_POSIX_MQUEUE_SYSCTL=y
CONFIG_IPC_NS=y
CONFIG_PID_NS=y
# CONFIG_BRIDGE_NETFILTER is not set
CONFIG_NETFILTER_XT_MATCH_ADDRTYPE=y
CONFIG_DEVTMPFS=y
# CONFIG_DEVTMPFS_MOUNT is not set
# CONFIG_DEVTMPFS_SAFE is not set
# CONFIG_VXLAN is not set


In [16]:
%cd ~/lamu-kernel
# Hash de version que hay que buscar despues en el pstore/uname del telefono.
# Como el paso 5 commiteo el cambio, este hash ahora SI es unico para esta variante de config.
!zcat out/mgk_64_k66/dist/Image.gz | strings | grep "Linux version" | head -1
!echo "---git HEAD actual (debe ser distinto al de builds anteriores con otro config)---"
!git -C kernel-6.6 rev-parse --short HEAD


/root/lamu-kernel
Linux version 6.6.127-android15-8-g98e7be363fe2-dirty-4k (kleaf@build-host) (Android (11368308, +pgo, +bolt, +lto, +mlgo, based on r510928) clang version 18.0.0 (https://android.googlesource.com/toolchain/llvm-project 477610d4d0d988e69dbc3fae4fe86bff3f07f2b5), LLD 18.0.0) #1 SMP PREEMPT Fri Aug  7 01:33:44 UTC 2026
---git HEAD actual (debe ser distinto al de builds anteriores con otro config)---
98e7be363


In [17]:
%%bash
cd ~/lamu-kernel
git -C kernel-6.6 rev-parse --short HEAD
zcat out/mgk_64_k66/dist/Image.gz | strings | grep "Linux version" | head -1

98e7be363
Linux version 6.6.127-android15-8-g98e7be363fe2-dirty-4k (kleaf@build-host) (Android (11368308, +pgo, +bolt, +lto, +mlgo, based on r510928) clang version 18.0.0 (https://android.googlesource.com/toolchain/llvm-project 477610d4d0d988e69dbc3fae4fe86bff3f07f2b5), LLD 18.0.0) #1 SMP PREEMPT Fri Aug  7 01:33:44 UTC 2026


## 8. Empaquetar boot.img

`ramdisk size: 0` es normal en este dispositivo GKI (el ramdisk real vive en `vendor_boot.img`).

In [18]:
%cd ~/lamu-kernel
!wget -O boot.img https://mirrorbits.lineageos.org/full/lamu/20260806/boot.img

!python3 tools/mkbootimg/unpack_bootimg.py --boot_img boot.img --out unpacked
!cp out/mgk_64_k66/dist/Image.gz unpacked/kernel

!python3 tools/mkbootimg/mkbootimg.py \
  --header_version 4 \
  --os_version 16.0.0 \
  --os_patch_level 2026-07 \
  --kernel unpacked/kernel \
  --ramdisk unpacked/ramdisk \
  --cmdline "" \
  --output boot_new.img

!./prebuilts/kernel-build-tools/linux-x86/bin/avbtool add_hash_footer \
  --image boot_new.img \
  --partition_size 67108864 \
  --partition_name boot \
  --algorithm NONE


/root/lamu-kernel
--2026-08-07 02:25:00--  https://mirrorbits.lineageos.org/full/lamu/20260806/boot.img
Resolving mirrorbits.lineageos.org (mirrorbits.lineageos.org)... 116.202.242.188
Connecting to mirrorbits.lineageos.org (mirrorbits.lineageos.org)|116.202.242.188|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://mirrors.ocf.berkeley.edu/lineageos/full/lamu/20260806/boot.img [following]
--2026-08-07 02:25:00--  https://mirrors.ocf.berkeley.edu/lineageos/full/lamu/20260806/boot.img
Resolving mirrors.ocf.berkeley.edu (mirrors.ocf.berkeley.edu)... 169.229.200.70, 2607:f140:0:32::70
Connecting to mirrors.ocf.berkeley.edu (mirrors.ocf.berkeley.edu)|169.229.200.70|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 67108864 (64M) [application/octet-stream]
Saving to: ‘boot.img’

boot.img            100%[===================>]  64.00M  20.0MB/s    in 4.0s    

2026-08-07 02:25:05 (16.1 MB/s) - ‘boot.img’ saved [67108864/67108864]

bo

## 9. Verificar el paquete final antes de descargar

In [19]:
%cd ~/lamu-kernel
!python3 tools/mkbootimg/unpack_bootimg.py --boot_img boot_new.img --out verify
!sha1sum verify/kernel out/mgk_64_k66/dist/Image.gz
# Los dos hashes DEBEN coincidir. Si no coinciden, no descargar/flashear todavia.


/root/lamu-kernel
boot magic: ANDROID!
kernel_size: 14862102
ramdisk size: 0
os version: 16.0.0
os patch level: 2026-07
boot image header version: 4
command line args: 
boot.img signature size: 0
b2da1736378d093856f7bcae6ba655a1ce31239b  verify/kernel
b2da1736378d093856f7bcae6ba655a1ce31239b  out/mgk_64_k66/dist/Image.gz


In [20]:
from google.colab import files
files.download("boot_new.img")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>